## split 1--deepseek_v3  ...pre_tokenizer

In [1]:
# code the split 1.
import regex

corpus = """
I bought 5 apples and 12 oranges today
东京塔 is 333 meters tall
彼女は3冊の本を持っている
The meeting is scheduled for 7pm in 会議室
价格是99元 for the ticket
She has 2 cats and 1 dog
この漫画は42巻まで出版されている
北京大学 was founded in year 189
Room 101 is next to 図書館
总共有365天 in a year
"""

In [2]:
documents = [line for line in corpus.split("\n") if line.strip()]
documents

['I bought 5 apples and 12 oranges today',
 '东京塔 is 333 meters tall',
 '彼女は3冊の本を持っている',
 'The meeting is scheduled for 7pm in 会議室',
 '价格是99元 for the ticket',
 'She has 2 cats and 1 dog',
 'この漫画は42巻まで出版されている',
 '北京大学 was founded in year 189',
 'Room 101 is next to 図書館',
 '总共有365天 in a year']

In [3]:
pattern = regex.compile(r"\p{N}{1,3}") # deepseekv3 tokenzier--pre_tokenize split#1
pattern

regex.Regex('\\p{N}{1,3}', flags=regex.V0)

In [4]:

lst = [[index, match.group(), match.span()] for index, document in enumerate(documents) for match in pattern.finditer(document)]
lst


[[0, '5', (9, 10)],
 [0, '12', (22, 24)],
 [1, '333', (7, 10)],
 [2, '3', (3, 4)],
 [3, '7', (29, 30)],
 [4, '99', (3, 5)],
 [5, '2', (8, 9)],
 [5, '1', (19, 20)],
 [6, '42', (5, 7)],
 [7, '189', (25, 28)],
 [8, '101', (5, 8)],
 [9, '365', (3, 6)]]

In [5]:
indices_to_change = [index[0] for index in lst]
start_end = [index[2] for index in lst]


In [6]:
indices_to_change

[0, 0, 1, 2, 3, 4, 5, 5, 6, 7, 8, 9]

In [7]:
start_end

[(9, 10),
 (22, 24),
 (7, 10),
 (3, 4),
 (29, 30),
 (3, 5),
 (8, 9),
 (19, 20),
 (5, 7),
 (25, 28),
 (5, 8),
 (3, 6)]

In [8]:
start, end = start_end[1]
start, end

(22, 24)

In [9]:
len(start_end[0])

2

In [10]:

def split1_transform(documents, indices_to_change, start_end):

    transformed_documents = []
    span_index = 0
    for document_index, document in enumerate(documents):

        temp_doc = []
        previous_end = 0
        while (span_index < len(start_end) and document_index == indices_to_change[span_index]):
            start, end = start_end[span_index]

            if start > previous_end:
                temp_doc.append(document[previous_end: start])
                previous_end = start

            temp_doc.append(document[start:end])
            previous_end = end

            span_index += 1

        if previous_end < len(document):
            temp_doc.append(document[previous_end:])

        transformed_documents.append(temp_doc)


    return transformed_documents

    

In [11]:
split1_transformed_data = split1_transform(
    documents,
    indices_to_change,
    start_end,
)
split1_transformed_data

[['I bought ', '5', ' apples and ', '12', ' oranges today'],
 ['东京塔 is ', '333', ' meters tall'],
 ['彼女は', '3', '冊の本を持っている'],
 ['The meeting is scheduled for ', '7', 'pm in 会議室'],
 ['价格是', '99', '元 for the ticket'],
 ['She has ', '2', ' cats and ', '1', ' dog'],
 ['この漫画は', '42', '巻まで出版されている'],
 ['北京大学 was founded in year ', '189'],
 ['Room ', '101', ' is next to 図書館'],
 ['总共有', '365', '天 in a year']]

In [12]:
split1_transformed_data[9][2]

'天 in a year'

In [13]:
for i in split1_transformed_data:
    print(i)

['I bought ', '5', ' apples and ', '12', ' oranges today']
['东京塔 is ', '333', ' meters tall']
['彼女は', '3', '冊の本を持っている']
['The meeting is scheduled for ', '7', 'pm in 会議室']
['价格是', '99', '元 for the ticket']
['She has ', '2', ' cats and ', '1', ' dog']
['この漫画は', '42', '巻まで出版されている']
['北京大学 was founded in year ', '189']
['Room ', '101', ' is next to 図書館']
['总共有', '365', '天 in a year']


## split 2

In [14]:
for document in split1_transformed_data:
    for piece in document:
        print(piece)
    break

I bought 
5
 apples and 
12
 oranges today


In [15]:
import regex

cjk_pattern = regex.compile("[一-龥\u3040-\u309F\u30A0-\u30FF]+")
cjk_pattern

regex.Regex('[一-龥\u3040-ゟ゠-ヿ]+', flags=regex.V0)

In [16]:

def split2_transform(split1_documents):

    transformed_documents = []

    for document in split1_documents:
        temp_doc = []

        for piece in document:
            previous_end = 0

            for match in cjk_pattern.finditer(piece):
                start, end = match.span()

                if start > previous_end:
                    temp_doc.append(piece[previous_end:start])

                temp_doc.append(piece[start:end])
                previous_end = end


            if previous_end < len(piece):
                temp_doc.append(piece[previous_end:])

        transformed_documents.append(temp_doc)

    return transformed_documents
                    

In [17]:
split2_transformed_data = split2_transform(split1_transformed_data)
split2_transformed_data

[['I bought ', '5', ' apples and ', '12', ' oranges today'],
 ['东京塔', ' is ', '333', ' meters tall'],
 ['彼女は', '3', '冊の本を持っている'],
 ['The meeting is scheduled for ', '7', 'pm in ', '会議室'],
 ['价格是', '99', '元', ' for the ticket'],
 ['She has ', '2', ' cats and ', '1', ' dog'],
 ['この漫画は', '42', '巻まで出版されている'],
 ['北京大学', ' was founded in year ', '189'],
 ['Room ', '101', ' is next to ', '図書館'],
 ['总共有', '365', '天', ' in a year']]

## Split 3

In [18]:
split3_pattern = regex.compile(
    r"""[!"#$%&'()*+,\-./:;<=>?@\[\\\]^_`{|}~][A-Za-z]+
|[^\r\n\p{L}\p{P}\p{S}]?[\p{L}\p{M}]+
| ?[\p{P}\p{S}]+[\r\n]*
|\s*[\r\n]+
|\s+(?!\S)
|\s+"""
)

In [19]:

def split3_transform(split2_transformed_data:list[list[str]]):

    transformed_documents = []

    for document in split2_transformed_data:
        temp_doc = []

        for piece in document:
            previous_end = 0

            for match in split3_pattern.finditer(piece):
                start, end = match.span()

                if start > previous_end:
                    temp_doc.append(piece[previous_end:start])


                temp_doc.append(piece[start:end])
                previous_end = end 

            if previous_end < len(piece):
                temp_doc.append(piece[previous_end:])

        transformed_documents.append(temp_doc)

    return transformed_documents



In [20]:
split3_transformed_data = split3_transform(split2_transformed_data)
split3_transformed_data

[['I',
  ' ',
  'bought',
  ' ',
  '5',
  ' ',
  'apples',
  ' ',
  'and',
  ' ',
  '12',
  ' ',
  'oranges',
  ' ',
  'today'],
 ['东京塔', ' ', 'is', ' ', '333', ' ', 'meters', ' ', 'tall'],
 ['彼女は', '3', '冊の本を持っている'],
 ['The',
  ' ',
  'meeting',
  ' ',
  'is',
  ' ',
  'scheduled',
  ' ',
  'for',
  ' ',
  '7',
  'pm',
  ' ',
  'in',
  ' ',
  '会議室'],
 ['价格是', '99', '元', ' ', 'for', ' ', 'the', ' ', 'ticket'],
 ['She', ' ', 'has', ' ', '2', ' ', 'cats', ' ', 'and', ' ', '1', ' ', 'dog'],
 ['この漫画は', '42', '巻まで出版されている'],
 ['北京大学', ' ', 'was', ' ', 'founded', ' ', 'in', ' ', 'year', ' ', '189'],
 ['Room', ' ', '101', ' ', 'is', ' ', 'next', ' ', 'to', ' ', '図書館'],
 ['总共有', '365', '天', ' ', 'in', ' ', 'a', ' ', 'year']]

In [21]:
for document in split3_transformed_data:
    print(document)

['I', ' ', 'bought', ' ', '5', ' ', 'apples', ' ', 'and', ' ', '12', ' ', 'oranges', ' ', 'today']
['东京塔', ' ', 'is', ' ', '333', ' ', 'meters', ' ', 'tall']
['彼女は', '3', '冊の本を持っている']
['The', ' ', 'meeting', ' ', 'is', ' ', 'scheduled', ' ', 'for', ' ', '7', 'pm', ' ', 'in', ' ', '会議室']
['价格是', '99', '元', ' ', 'for', ' ', 'the', ' ', 'ticket']
['She', ' ', 'has', ' ', '2', ' ', 'cats', ' ', 'and', ' ', '1', ' ', 'dog']
['この漫画は', '42', '巻まで出版されている']
['北京大学', ' ', 'was', ' ', 'founded', ' ', 'in', ' ', 'year', ' ', '189']
['Room', ' ', '101', ' ', 'is', ' ', 'next', ' ', 'to', ' ', '図書館']
['总共有', '365', '天', ' ', 'in', ' ', 'a', ' ', 'year']


#### splitting done

## byte-level conversion

In [22]:

#  taking split-3 transformed documents.

def byte_conversion(split3_transformed_data:list[list[str]]) -> list[list[int]]:

    document_bytes = []
    for document in split3_transformed_data:

        piece_bytes = []
        for piece in document:

            
            for element in piece:
                piece_bytes.append(element.encode("utf-8"))


        document_bytes.append(piece_bytes)

    return  document_bytes


In [23]:
result = byte_conversion(split3_transformed_data)

for i in result:
    print(i)

[b'I', b' ', b'b', b'o', b'u', b'g', b'h', b't', b' ', b'5', b' ', b'a', b'p', b'p', b'l', b'e', b's', b' ', b'a', b'n', b'd', b' ', b'1', b'2', b' ', b'o', b'r', b'a', b'n', b'g', b'e', b's', b' ', b't', b'o', b'd', b'a', b'y']
[b'\xe4\xb8\x9c', b'\xe4\xba\xac', b'\xe5\xa1\x94', b' ', b'i', b's', b' ', b'3', b'3', b'3', b' ', b'm', b'e', b't', b'e', b'r', b's', b' ', b't', b'a', b'l', b'l']
[b'\xe5\xbd\xbc', b'\xe5\xa5\xb3', b'\xe3\x81\xaf', b'3', b'\xe5\x86\x8a', b'\xe3\x81\xae', b'\xe6\x9c\xac', b'\xe3\x82\x92', b'\xe6\x8c\x81', b'\xe3\x81\xa3', b'\xe3\x81\xa6', b'\xe3\x81\x84', b'\xe3\x82\x8b']
[b'T', b'h', b'e', b' ', b'm', b'e', b'e', b't', b'i', b'n', b'g', b' ', b'i', b's', b' ', b's', b'c', b'h', b'e', b'd', b'u', b'l', b'e', b'd', b' ', b'f', b'o', b'r', b' ', b'7', b'p', b'm', b' ', b'i', b'n', b' ', b'\xe4\xbc\x9a', b'\xe8\xad\xb0', b'\xe5\xae\xa4']
[b'\xe4\xbb\xb7', b'\xe6\xa0\xbc', b'\xe6\x98\xaf', b'9', b'9', b'\xe5\x85\x83', b' ', b'f', b'o', b'r', b' ', b't', b'h', b'e

In [24]:
for document in result[0]:
    for piece in document:
        print(piece)

73
32
98
111
117
103
104
116
32
53
32
97
112
112
108
101
115
32
97
110
100
32
49
50
32
111
114
97
110
103
101
115
32
116
111
100
97
121


In [25]:

all_bytes_list = []
for document in result:
    temp_list = []
    for piece in document:
        for i in piece:
            temp_list.append(i)

    all_bytes_list.append(temp_list)
    

In [26]:
for i in all_bytes_list:
    print(i)

[73, 32, 98, 111, 117, 103, 104, 116, 32, 53, 32, 97, 112, 112, 108, 101, 115, 32, 97, 110, 100, 32, 49, 50, 32, 111, 114, 97, 110, 103, 101, 115, 32, 116, 111, 100, 97, 121]
[228, 184, 156, 228, 186, 172, 229, 161, 148, 32, 105, 115, 32, 51, 51, 51, 32, 109, 101, 116, 101, 114, 115, 32, 116, 97, 108, 108]
[229, 189, 188, 229, 165, 179, 227, 129, 175, 51, 229, 134, 138, 227, 129, 174, 230, 156, 172, 227, 130, 146, 230, 140, 129, 227, 129, 163, 227, 129, 166, 227, 129, 132, 227, 130, 139]
[84, 104, 101, 32, 109, 101, 101, 116, 105, 110, 103, 32, 105, 115, 32, 115, 99, 104, 101, 100, 117, 108, 101, 100, 32, 102, 111, 114, 32, 55, 112, 109, 32, 105, 110, 32, 228, 188, 154, 232, 173, 176, 229, 174, 164]
[228, 187, 183, 230, 160, 188, 230, 152, 175, 57, 57, 229, 133, 131, 32, 102, 111, 114, 32, 116, 104, 101, 32, 116, 105, 99, 107, 101, 116]
[83, 104, 101, 32, 104, 97, 115, 32, 50, 32, 99, 97, 116, 115, 32, 97, 110, 100, 32, 49, 32, 100, 111, 103]
[227, 129, 147, 227, 129, 174, 230, 188, 17

#### byte-conversion done

In [27]:
# build byte representation mapping

from tokenizers.normalizers import ByteLevel
normalizer = ByteLevel()
normalizer.normalize_str("😀")

'ðŁĺĢ'

In [28]:
bs = []

for i in range(33,127):
    bs.append(i)

In [29]:
for i in range(161,173):
    bs.append(i)

In [30]:
for i in range(174, 256):
    bs.append(i)

In [31]:
bs

[33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 185,
 186,
 187,
 188,
 189,
 190,
 191,
 192,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 205,
 206,
 207,
 208,
 209,
 210,
 211,
 212,
 213,
 214,
 215,
 216,
 217,
 218,
 219,
 220,
 221,
 222,
 223,
 224,
 225,
 226,
 227,
 228,
 229,
 230,
 231,
 232,
 233,
 234,
 235,
 236,
 237,
 238,
 239,
 240,
 241,
 242,
 243,
 244,
 245,

In [32]:
len(bs)

188

In [33]:
unicode_repr_non_missing = []
for i in bs:
    unicode_repr_non_missing.append(chr(i))

In [34]:
unicode_repr_non_missing

['!',
 '"',
 '#',
 '$',
 '%',
 '&',
 "'",
 '(',
 ')',
 '*',
 '+',
 ',',
 '-',
 '.',
 '/',
 '0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 ':',
 ';',
 '<',
 '=',
 '>',
 '?',
 '@',
 'A',
 'B',
 'C',
 'D',
 'E',
 'F',
 'G',
 'H',
 'I',
 'J',
 'K',
 'L',
 'M',
 'N',
 'O',
 'P',
 'Q',
 'R',
 'S',
 'T',
 'U',
 'V',
 'W',
 'X',
 'Y',
 'Z',
 '[',
 '\\',
 ']',
 '^',
 '_',
 '`',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z',
 '{',
 '|',
 '}',
 '~',
 '¡',
 '¢',
 '£',
 '¤',
 '¥',
 '¦',
 '§',
 '¨',
 '©',
 'ª',
 '«',
 '¬',
 '®',
 '¯',
 '°',
 '±',
 '²',
 '³',
 '´',
 'µ',
 '¶',
 '·',
 '¸',
 '¹',
 'º',
 '»',
 '¼',
 '½',
 '¾',
 '¿',
 'À',
 'Á',
 'Â',
 'Ã',
 'Ä',
 'Å',
 'Æ',
 'Ç',
 'È',
 'É',
 'Ê',
 'Ë',
 'Ì',
 'Í',
 'Î',
 'Ï',
 'Ð',
 'Ñ',
 'Ò',
 'Ó',
 'Ô',
 'Õ',
 'Ö',
 '×',
 'Ø',
 'Ù',
 'Ú',
 'Û',
 'Ü',
 'Ý',
 'Þ',
 'ß',
 'à',
 'á',
 'â',
 'ã',
 'ä',
 'å',
 'æ',
 'ç',
 'è',
 'é',
 'ê

In [35]:
len(unicode_repr_non_missing)

188

In [36]:
# missing chars.

missing_bytes = []
for i in range(256, 324):
    missing_bytes.append(i)
    

In [37]:
missing_bytes

[256,
 257,
 258,
 259,
 260,
 261,
 262,
 263,
 264,
 265,
 266,
 267,
 268,
 269,
 270,
 271,
 272,
 273,
 274,
 275,
 276,
 277,
 278,
 279,
 280,
 281,
 282,
 283,
 284,
 285,
 286,
 287,
 288,
 289,
 290,
 291,
 292,
 293,
 294,
 295,
 296,
 297,
 298,
 299,
 300,
 301,
 302,
 303,
 304,
 305,
 306,
 307,
 308,
 309,
 310,
 311,
 312,
 313,
 314,
 315,
 316,
 317,
 318,
 319,
 320,
 321,
 322,
 323]

In [38]:
len(missing_bytes)

68

In [39]:
missing_bytes_unicode_representation = []
for  i in missing_bytes:
    missing_bytes_unicode_representation.append(chr(i))

In [40]:
missing_bytes_unicode_representation

['Ā',
 'ā',
 'Ă',
 'ă',
 'Ą',
 'ą',
 'Ć',
 'ć',
 'Ĉ',
 'ĉ',
 'Ċ',
 'ċ',
 'Č',
 'č',
 'Ď',
 'ď',
 'Đ',
 'đ',
 'Ē',
 'ē',
 'Ĕ',
 'ĕ',
 'Ė',
 'ė',
 'Ę',
 'ę',
 'Ě',
 'ě',
 'Ĝ',
 'ĝ',
 'Ğ',
 'ğ',
 'Ġ',
 'ġ',
 'Ģ',
 'ģ',
 'Ĥ',
 'ĥ',
 'Ħ',
 'ħ',
 'Ĩ',
 'ĩ',
 'Ī',
 'ī',
 'Ĭ',
 'ĭ',
 'Į',
 'į',
 'İ',
 'ı',
 'Ĳ',
 'ĳ',
 'Ĵ',
 'ĵ',
 'Ķ',
 'ķ',
 'ĸ',
 'Ĺ',
 'ĺ',
 'Ļ',
 'ļ',
 'Ľ',
 'ľ',
 'Ŀ',
 'ŀ',
 'Ł',
 'ł',
 'Ń']

In [41]:
len(missing_bytes_unicode_representation)

68

In [42]:
bytes = []

for i in range(0,256):
    bytes.append(i)

In [43]:
len(bytes)

256

In [44]:
# compiling everything here in dictionary

def compiling_bytes(byte_values, unicode_repr_non_missing, missing_bytes_unicode_representation):

    hashmap = {}
    safe_unicode_pointer = 0
    missing_unicode_pointer = 0
    # loop over bytes
    for byte in byte_values:
        safe_search = 33 <= byte <= 126
        safe_search_extended = 161 <= byte <= 172 or 174 <= byte <= 255

        if safe_search or safe_search_extended:
            hashmap[byte] = unicode_repr_non_missing[safe_unicode_pointer]
            safe_unicode_pointer += 1
        else:
            hashmap[byte] = missing_bytes_unicode_representation[missing_unicode_pointer]
            missing_unicode_pointer += 1

    return hashmap

In [45]:
byte_level_symbols = compiling_bytes(bytes,  unicode_repr_non_missing, missing_bytes_unicode_representation)
byte_level_symbols

{0: 'Ā',
 1: 'ā',
 2: 'Ă',
 3: 'ă',
 4: 'Ą',
 5: 'ą',
 6: 'Ć',
 7: 'ć',
 8: 'Ĉ',
 9: 'ĉ',
 10: 'Ċ',
 11: 'ċ',
 12: 'Č',
 13: 'č',
 14: 'Ď',
 15: 'ď',
 16: 'Đ',
 17: 'đ',
 18: 'Ē',
 19: 'ē',
 20: 'Ĕ',
 21: 'ĕ',
 22: 'Ė',
 23: 'ė',
 24: 'Ę',
 25: 'ę',
 26: 'Ě',
 27: 'ě',
 28: 'Ĝ',
 29: 'ĝ',
 30: 'Ğ',
 31: 'ğ',
 32: 'Ġ',
 33: '!',
 34: '"',
 35: '#',
 36: '$',
 37: '%',
 38: '&',
 39: "'",
 40: '(',
 41: ')',
 42: '*',
 43: '+',
 44: ',',
 45: '-',
 46: '.',
 47: '/',
 48: '0',
 49: '1',
 50: '2',
 51: '3',
 52: '4',
 53: '5',
 54: '6',
 55: '7',
 56: '8',
 57: '9',
 58: ':',
 59: ';',
 60: '<',
 61: '=',
 62: '>',
 63: '?',
 64: '@',
 65: 'A',
 66: 'B',
 67: 'C',
 68: 'D',
 69: 'E',
 70: 'F',
 71: 'G',
 72: 'H',
 73: 'I',
 74: 'J',
 75: 'K',
 76: 'L',
 77: 'M',
 78: 'N',
 79: 'O',
 80: 'P',
 81: 'Q',
 82: 'R',
 83: 'S',
 84: 'T',
 85: 'U',
 86: 'V',
 87: 'W',
 88: 'X',
 89: 'Y',
 90: 'Z',
 91: '[',
 92: '\\',
 93: ']',
 94: '^',
 95: '_',
 96: '`',
 97: 'a',
 98: 'b',
 99: 'c',
 100: 'd'

In [47]:
for i in all_bytes_list:
    print(i)

[73, 32, 98, 111, 117, 103, 104, 116, 32, 53, 32, 97, 112, 112, 108, 101, 115, 32, 97, 110, 100, 32, 49, 50, 32, 111, 114, 97, 110, 103, 101, 115, 32, 116, 111, 100, 97, 121]
[228, 184, 156, 228, 186, 172, 229, 161, 148, 32, 105, 115, 32, 51, 51, 51, 32, 109, 101, 116, 101, 114, 115, 32, 116, 97, 108, 108]
[229, 189, 188, 229, 165, 179, 227, 129, 175, 51, 229, 134, 138, 227, 129, 174, 230, 156, 172, 227, 130, 146, 230, 140, 129, 227, 129, 163, 227, 129, 166, 227, 129, 132, 227, 130, 139]
[84, 104, 101, 32, 109, 101, 101, 116, 105, 110, 103, 32, 105, 115, 32, 115, 99, 104, 101, 100, 117, 108, 101, 100, 32, 102, 111, 114, 32, 55, 112, 109, 32, 105, 110, 32, 228, 188, 154, 232, 173, 176, 229, 174, 164]
[228, 187, 183, 230, 160, 188, 230, 152, 175, 57, 57, 229, 133, 131, 32, 102, 111, 114, 32, 116, 104, 101, 32, 116, 105, 99, 107, 101, 116]
[83, 104, 101, 32, 104, 97, 115, 32, 50, 32, 99, 97, 116, 115, 32, 97, 110, 100, 32, 49, 32, 100, 111, 103]
[227, 129, 147, 227, 129, 174, 230, 188, 17

In [50]:
# getting byte-level symbols

def byte_level_representation(all_bytes_list:list[list[int]], byte_level_symbols: dict) -> list[list[str]]:

    overall_byte_symbol_representation = []
    for byte_document in all_bytes_list:
        temp_list = []
        for byte in byte_document:
        
            if byte in byte_level_symbols:
                symbol = byte_level_symbols[byte]
                temp_list.append(symbol)

        
        overall_byte_symbol_representation.append(temp_list)

    return overall_byte_symbol_representation 


In [51]:
overall_byte_symbol_representation = byte_level_representation(all_bytes_list, byte_level_symbols)
overall_byte_symbol_representation

[['I',
  'Ġ',
  'b',
  'o',
  'u',
  'g',
  'h',
  't',
  'Ġ',
  '5',
  'Ġ',
  'a',
  'p',
  'p',
  'l',
  'e',
  's',
  'Ġ',
  'a',
  'n',
  'd',
  'Ġ',
  '1',
  '2',
  'Ġ',
  'o',
  'r',
  'a',
  'n',
  'g',
  'e',
  's',
  'Ġ',
  't',
  'o',
  'd',
  'a',
  'y'],
 ['ä',
  '¸',
  'ľ',
  'ä',
  'º',
  '¬',
  'å',
  '¡',
  'Ķ',
  'Ġ',
  'i',
  's',
  'Ġ',
  '3',
  '3',
  '3',
  'Ġ',
  'm',
  'e',
  't',
  'e',
  'r',
  's',
  'Ġ',
  't',
  'a',
  'l',
  'l'],
 ['å',
  '½',
  '¼',
  'å',
  '¥',
  '³',
  'ã',
  'ģ',
  '¯',
  '3',
  'å',
  'Ĩ',
  'Ĭ',
  'ã',
  'ģ',
  '®',
  'æ',
  'ľ',
  '¬',
  'ã',
  'Ĥ',
  'Ĵ',
  'æ',
  'Į',
  'ģ',
  'ã',
  'ģ',
  '£',
  'ã',
  'ģ',
  '¦',
  'ã',
  'ģ',
  'Ħ',
  'ã',
  'Ĥ',
  'ĭ'],
 ['T',
  'h',
  'e',
  'Ġ',
  'm',
  'e',
  'e',
  't',
  'i',
  'n',
  'g',
  'Ġ',
  'i',
  's',
  'Ġ',
  's',
  'c',
  'h',
  'e',
  'd',
  'u',
  'l',
  'e',
  'd',
  'Ġ',
  'f',
  'o',
  'r',
  'Ġ',
  '7',
  'p',
  'm',
  'Ġ',
  'i',
  'n',
  'Ġ',
  'ä',
  '¼',
  'ļ',
  '

In [52]:
#
for i in overall_byte_symbol_representation:
    print(i)

['I', 'Ġ', 'b', 'o', 'u', 'g', 'h', 't', 'Ġ', '5', 'Ġ', 'a', 'p', 'p', 'l', 'e', 's', 'Ġ', 'a', 'n', 'd', 'Ġ', '1', '2', 'Ġ', 'o', 'r', 'a', 'n', 'g', 'e', 's', 'Ġ', 't', 'o', 'd', 'a', 'y']
['ä', '¸', 'ľ', 'ä', 'º', '¬', 'å', '¡', 'Ķ', 'Ġ', 'i', 's', 'Ġ', '3', '3', '3', 'Ġ', 'm', 'e', 't', 'e', 'r', 's', 'Ġ', 't', 'a', 'l', 'l']
['å', '½', '¼', 'å', '¥', '³', 'ã', 'ģ', '¯', '3', 'å', 'Ĩ', 'Ĭ', 'ã', 'ģ', '®', 'æ', 'ľ', '¬', 'ã', 'Ĥ', 'Ĵ', 'æ', 'Į', 'ģ', 'ã', 'ģ', '£', 'ã', 'ģ', '¦', 'ã', 'ģ', 'Ħ', 'ã', 'Ĥ', 'ĭ']
['T', 'h', 'e', 'Ġ', 'm', 'e', 'e', 't', 'i', 'n', 'g', 'Ġ', 'i', 's', 'Ġ', 's', 'c', 'h', 'e', 'd', 'u', 'l', 'e', 'd', 'Ġ', 'f', 'o', 'r', 'Ġ', '7', 'p', 'm', 'Ġ', 'i', 'n', 'Ġ', 'ä', '¼', 'ļ', 'è', 'Ń', '°', 'å', '®', '¤']
['ä', '»', '·', 'æ', 'ł', '¼', 'æ', 'ĺ', '¯', '9', '9', 'å', 'ħ', 'ĥ', 'Ġ', 'f', 'o', 'r', 'Ġ', 't', 'h', 'e', 'Ġ', 't', 'i', 'c', 'k', 'e', 't']
['S', 'h', 'e', 'Ġ', 'h', 'a', 's', 'Ġ', '2', 'Ġ', 'c', 'a', 't', 's', 'Ġ', 'a', 'n', 'd', 'Ġ', '1', 'Ġ', 'd',

In [55]:
for document in overall_byte_symbol_representation:
    temp_list = []
    i = 0
    while i < len(document) - 1:
        pair = document[i],  document[i+1]
        temp_list.append(pair)
        i += 1
    print(temp_list)

        
    break

[('I', 'Ġ'), ('Ġ', 'b'), ('b', 'o'), ('o', 'u'), ('u', 'g'), ('g', 'h'), ('h', 't'), ('t', 'Ġ'), ('Ġ', '5'), ('5', 'Ġ'), ('Ġ', 'a'), ('a', 'p'), ('p', 'p'), ('p', 'l'), ('l', 'e'), ('e', 's'), ('s', 'Ġ'), ('Ġ', 'a'), ('a', 'n'), ('n', 'd'), ('d', 'Ġ'), ('Ġ', '1'), ('1', '2'), ('2', 'Ġ'), ('Ġ', 'o'), ('o', 'r'), ('r', 'a'), ('a', 'n'), ('n', 'g'), ('g', 'e'), ('e', 's'), ('s', 'Ġ'), ('Ġ', 't'), ('t', 'o'), ('o', 'd'), ('d', 'a'), ('a', 'y')]


In [56]:

## get everything in pairs

def create_pairs(overall_byte_symbol_representation:list[list[str]]) -> list[list[tuple]]:

    universal_list = []
    for document in overall_byte_symbol_representation:

        temp_list = []
        i = 0
        while i < len(document) - 1:
            pair = document[i], document[i+1]
            temp_list.append(pair)
            i += 1

        universal_list.append(temp_list)

    return universal_list


In [57]:
universal_symbol_pairs = create_pairs(overall_byte_symbol_representation)
universal_symbol_pairs

[[('I', 'Ġ'),
  ('Ġ', 'b'),
  ('b', 'o'),
  ('o', 'u'),
  ('u', 'g'),
  ('g', 'h'),
  ('h', 't'),
  ('t', 'Ġ'),
  ('Ġ', '5'),
  ('5', 'Ġ'),
  ('Ġ', 'a'),
  ('a', 'p'),
  ('p', 'p'),
  ('p', 'l'),
  ('l', 'e'),
  ('e', 's'),
  ('s', 'Ġ'),
  ('Ġ', 'a'),
  ('a', 'n'),
  ('n', 'd'),
  ('d', 'Ġ'),
  ('Ġ', '1'),
  ('1', '2'),
  ('2', 'Ġ'),
  ('Ġ', 'o'),
  ('o', 'r'),
  ('r', 'a'),
  ('a', 'n'),
  ('n', 'g'),
  ('g', 'e'),
  ('e', 's'),
  ('s', 'Ġ'),
  ('Ġ', 't'),
  ('t', 'o'),
  ('o', 'd'),
  ('d', 'a'),
  ('a', 'y')],
 [('ä', '¸'),
  ('¸', 'ľ'),
  ('ľ', 'ä'),
  ('ä', 'º'),
  ('º', '¬'),
  ('¬', 'å'),
  ('å', '¡'),
  ('¡', 'Ķ'),
  ('Ķ', 'Ġ'),
  ('Ġ', 'i'),
  ('i', 's'),
  ('s', 'Ġ'),
  ('Ġ', '3'),
  ('3', '3'),
  ('3', '3'),
  ('3', 'Ġ'),
  ('Ġ', 'm'),
  ('m', 'e'),
  ('e', 't'),
  ('t', 'e'),
  ('e', 'r'),
  ('r', 's'),
  ('s', 'Ġ'),
  ('Ġ', 't'),
  ('t', 'a'),
  ('a', 'l'),
  ('l', 'l')],
 [('å', '½'),
  ('½', '¼'),
  ('¼', 'å'),
  ('å', '¥'),
  ('¥', '³'),
  ('³', 'ã'),
  ('ã', 'ģ'),
  ('

In [ ]:
for i in universal_symbol_pairs:  # got the pairs
    print(i)

[('I', 'Ġ'), ('Ġ', 'b'), ('b', 'o'), ('o', 'u'), ('u', 'g'), ('g', 'h'), ('h', 't'), ('t', 'Ġ'), ('Ġ', '5'), ('5', 'Ġ'), ('Ġ', 'a'), ('a', 'p'), ('p', 'p'), ('p', 'l'), ('l', 'e'), ('e', 's'), ('s', 'Ġ'), ('Ġ', 'a'), ('a', 'n'), ('n', 'd'), ('d', 'Ġ'), ('Ġ', '1'), ('1', '2'), ('2', 'Ġ'), ('Ġ', 'o'), ('o', 'r'), ('r', 'a'), ('a', 'n'), ('n', 'g'), ('g', 'e'), ('e', 's'), ('s', 'Ġ'), ('Ġ', 't'), ('t', 'o'), ('o', 'd'), ('d', 'a'), ('a', 'y')]
[('ä', '¸'), ('¸', 'ľ'), ('ľ', 'ä'), ('ä', 'º'), ('º', '¬'), ('¬', 'å'), ('å', '¡'), ('¡', 'Ķ'), ('Ķ', 'Ġ'), ('Ġ', 'i'), ('i', 's'), ('s', 'Ġ'), ('Ġ', '3'), ('3', '3'), ('3', '3'), ('3', 'Ġ'), ('Ġ', 'm'), ('m', 'e'), ('e', 't'), ('t', 'e'), ('e', 'r'), ('r', 's'), ('s', 'Ġ'), ('Ġ', 't'), ('t', 'a'), ('a', 'l'), ('l', 'l')]
[('å', '½'), ('½', '¼'), ('¼', 'å'), ('å', '¥'), ('¥', '³'), ('³', 'ã'), ('ã', 'ģ'), ('ģ', '¯'), ('¯', '3'), ('3', 'å'), ('å', 'Ĩ'), ('Ĩ', 'Ĭ'), ('Ĭ', 'ã'), ('ã', 'ģ'), ('ģ', '®'), ('®', 'æ'), ('æ', 'ľ'), ('ľ', '¬'), ('¬', 'ã'), (

In [92]:
hash  = {"age": 55, "height": 5.5}

for i in hash:
    print(hash[i])

55
5.5


In [78]:
# find the most frequent pairs

def pair_frequency_counter(universal_symbol_pairs: list[list[tuple]]) -> dict:

    hashmap = {}
    for document in universal_symbol_pairs:

        for pair in document:
            if pair in hashmap:
                hashmap[pair] += 1

            else:
                hashmap[pair] = 1

    return hashmap






    return hashmap


In [79]:
frequency_count_map = pair_frequency_counter(universal_symbol_pairs)
frequency_count_map

{('I', 'Ġ'): 1,
 ('Ġ', 'b'): 1,
 ('b', 'o'): 1,
 ('o', 'u'): 2,
 ('u', 'g'): 1,
 ('g', 'h'): 1,
 ('h', 't'): 1,
 ('t', 'Ġ'): 2,
 ('Ġ', '5'): 1,
 ('5', 'Ġ'): 1,
 ('Ġ', 'a'): 4,
 ('a', 'p'): 1,
 ('p', 'p'): 1,
 ('p', 'l'): 1,
 ('l', 'e'): 2,
 ('e', 's'): 2,
 ('s', 'Ġ'): 9,
 ('a', 'n'): 3,
 ('n', 'd'): 3,
 ('d', 'Ġ'): 4,
 ('Ġ', '1'): 4,
 ('1', '2'): 1,
 ('2', 'Ġ'): 2,
 ('Ġ', 'o'): 1,
 ('o', 'r'): 3,
 ('r', 'a'): 1,
 ('n', 'g'): 2,
 ('g', 'e'): 1,
 ('Ġ', 't'): 5,
 ('t', 'o'): 2,
 ('o', 'd'): 1,
 ('d', 'a'): 1,
 ('a', 'y'): 1,
 ('ä', '¸'): 1,
 ('¸', 'ľ'): 1,
 ('ľ', 'ä'): 1,
 ('ä', 'º'): 2,
 ('º', '¬'): 2,
 ('¬', 'å'): 2,
 ('å', '¡'): 1,
 ('¡', 'Ķ'): 1,
 ('Ķ', 'Ġ'): 1,
 ('Ġ', 'i'): 6,
 ('i', 's'): 3,
 ('Ġ', '3'): 1,
 ('3', '3'): 2,
 ('3', 'Ġ'): 1,
 ('Ġ', 'm'): 2,
 ('m', 'e'): 2,
 ('e', 't'): 3,
 ('t', 'e'): 1,
 ('e', 'r'): 1,
 ('r', 's'): 1,
 ('t', 'a'): 1,
 ('a', 'l'): 1,
 ('l', 'l'): 1,
 ('å', '½'): 1,
 ('½', '¼'): 1,
 ('¼', 'å'): 1,
 ('å', '¥'): 1,
 ('¥', '³'): 1,
 ('³', 'ã'): 1,
 ('ã', '

In [108]:

def get_highest_frequency_pair(frequency_count_map:dict) -> list:

    highest_count = 0
    which_pair = []

    for pair in frequency_count_map:
        if frequency_count_map[pair] > highest_count:
            highest_count = frequency_count_map[pair]
            which_pair.append(pair)


    return which_pair[-1], highest_count


In [109]:
pair, highest_count = get_highest_frequency_pair(frequency_count_map)
pair, highest_count

(('ã', 'ģ'), 13)

In [ ]:
# merge -- in overall_byte_symbol_representation